<a href="https://colab.research.google.com/github/Mariam-Elbishbeashy/HeadlineGeneration-NLP/blob/main/headlineGenerationNLP_flan_t5_small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📰 FLAN-T5 News Headline Generation Model

## 🎯 Objective
This project fine-tunes the **FLAN-T5-Small** model to generate short and meaningful news headlines from full articles using a text-to-text generation approach.

---

## Model Description

- **Base Model:** `google/flan-t5-small`
- **Architecture:** Encoder–Decoder Transformer
- **Model Type:** Text-to-Text Generation
- **Task:** Abstractive Headline Generation
- **Framework:** Hugging Face Transformers

The model is based on the T5 architecture, where both input and output are treated as text, making it highly flexible for NLP generation tasks.

---

## Training Configuration

- **Model:** FLAN-T5-Small
- **Trainer:** Seq2SeqTrainer
- **Loss Function:** Cross-Entropy Loss
- **Optimizer:** Adafactor
- **Learning Rate:** 2e-4
- **Batch Size:** 8
- **Epochs:** 2
- **Precision:** FP16 (GPU enabled)

---

## Evaluation Metrics

The model is evaluated using ROUGE scores:

- **ROUGE-1:** Word-level overlap
- **ROUGE-2:** Phrase-level overlap
- **ROUGE-L:** Sequence similarity

These metrics measure how close the generated headlines are to the reference headlines.

---

## Inference Strategy

- Input format: `"summarize: <article>"`
- Generation method: Beam Search
- Beam size: 4
- Output: Single generated headline per article

---

## Output

After training, the model is saved and reused for:
- Inference
- Real-time headline generation
- Deployment

In [ ]:

import os
import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

import evaluate

from google.colab import drive
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:", device)

if device == "cpu":
    print("⚠️ ENABLE GPU:")
    print("Runtime → Change runtime type → GPU")


DATA_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data"

train_df = pd.read_csv(
    os.path.join(DATA_DIR, "flan_t5_small_train.csv")
)[["model_input", "model_target"]].dropna()

val_df = pd.read_csv(
    os.path.join(DATA_DIR, "flan_t5_small_validation.csv")
)[["model_input", "model_target"]].dropna()

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "flan_t5_small_test.csv")
)[["model_input", "model_target"]].dropna()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

train_dataset = Dataset.from_pandas(
    train_df.reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df.reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df.reset_index(drop=True)
)

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.to(device)

MAX_INPUT = 256
MAX_OUTPUT = 32

def preprocess(batch):

    model_inputs = tokenizer(
        batch["model_input"],
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        batch["model_target"],
        max_length=MAX_OUTPUT,
        truncation=True,
        padding="max_length"
    )

    label_ids = labels["input_ids"]

    label_ids = [
        [
            token if token != tokenizer.pad_token_id else -100
            for token in label
        ]
        for label in label_ids
    ]

    model_inputs["labels"] = label_ids

    return model_inputs

train_dataset = train_dataset.map(
    preprocess,
    batched=True
)

val_dataset = val_dataset.map(
    preprocess,
    batched=True
)

test_dataset = test_dataset.map(
    preprocess,
    batched=True
)

train_dataset = train_dataset.remove_columns(
    ["model_input", "model_target"]
)

val_dataset = val_dataset.remove_columns(
    ["model_input", "model_target"]
)

test_dataset = test_dataset.remove_columns(
    ["model_input", "model_target"]
)

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.where(
        predictions != -100,
        predictions,
        tokenizer.pad_token_id
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        k: round(v, 4)
        for k, v in result.items()
    }

training_args = Seq2SeqTrainingArguments(

    output_dir="/content/flan_t5_results",

    evaluation_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-4,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    predict_with_generate=True,

    logging_steps=100,

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    optim="adafactor",

    report_to="none"
)

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

trainer.train()

print("\nTEST RESULTS")

results = trainer.evaluate(test_dataset)

print(results)

SAVE_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/flan_t5_small_model"

trainer.save_model(SAVE_DIR)

tokenizer.save_pretrained(SAVE_DIR)

print("\n✅ MODEL SAVED SUCCESSFULLY")
print(SAVE_DIR)

def generate_headline(text):

    model.eval()

    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_INPUT
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_length=32,
            num_beams=4,
            early_stopping=True
        )

    headline = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return headline

print("\n===================================")
print("HEADLINE GENERATOR READY")
print("Type exit to stop")
print("===================================\n")

while True:

    user_text = input("Enter Article:\n")

    if user_text.lower() == "exit":
        print("Stopped.")
        break

    prediction = generate_headline(user_text)

    print("\nGenerated Headline:")
    print(prediction)

    print("\n" + "="*60 + "\n")

# CHECK IF MODEL IS SAVED CORRECTLY


In [ ]:


import os

SAVE_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/flan_t5_small_model"

print("Checking folder...\n")

if os.path.exists(SAVE_DIR):

    print("✅ Model folder exists.\n")

    files = os.listdir(SAVE_DIR)

    print("Saved Files:")

    for file in files:
        print(file)

    required_files = [
        "config.json",
        "generation_config.json",
        "model.safetensors",
        "tokenizer_config.json",
        "special_tokens_map.json"
    ]

    print("\nChecking important files...\n")

    for file in required_files:

        if file in files:
            print(f"✅ {file} FOUND")
        else:
            print(f"❌ {file} MISSING")

else:
    print("❌ Model folder NOT found.")

Checking folder...

✅ Model folder exists.

Saved Files:
training_args.bin
config.json
generation_config.json
model.safetensors
tokenizer_config.json
special_tokens_map.json
spiece.model
tokenizer.json

Checking important files...

✅ config.json FOUND
✅ generation_config.json FOUND
✅ model.safetensors FOUND
✅ tokenizer_config.json FOUND
✅ special_tokens_map.json FOUND


# LOAD SAVED MODEL AND TEST


In [ ]:


from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

SAVE_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/flan_t5_small_model"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading saved model...\n")

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_DIR)

model.to(device)

print("✅ MODEL LOADED SUCCESSFULLY!")


text = """
The government announced major economic reforms to reduce inflation and improve growth.
"""

input_text = "summarize: " + text

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    max_length=256
).to(device)

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_length=32,
        num_beams=4
    )

headline = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\nGenerated Headline:")
print(headline)

Loading saved model...



Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ MODEL LOADED SUCCESSFULLY!

Generated Headline:
Government announces major economic reforms aimed at reducing inflation | TheHill's


# INTERACTIVE LOOP TEST


In [ ]:


def generate_headline(text):

    model.eval()

    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=256,
        truncation=True,
        padding="max_length"
    ).to(device)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_length=32,
            num_beams=4,
            early_stopping=True
        )

    headline = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return headline


print("\n🔥 FLAN-T5 Headline Generator Ready!")
print("Type 'exit' to stop.\n")

while True:

    user_text = input("Enter news text:\n")

    if user_text.lower() == "exit":
        print("Stopped.")
        break

    result = generate_headline(user_text)

    print("\n📰 Generated Headline:")
    print(result)

    print("\n" + "-" * 60 + "\n")


🔥 FLAN-T5 Headline Generator Ready!
Type 'exit' to stop.

Enter news text:
exit
Stopped.


# FLAN-T5 MODEL EVALUATION
# ROUGE + EXACT MATCH + SAMPLE PREDICTIONS

In [ ]:


import os
import pandas as pd
import numpy as np
import torch
import evaluate


DATA_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data"

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "t5_small_test.csv")
)[["model_input", "model_target"]].dropna()

print("Test Samples:", len(test_df))


rouge = evaluate.load("rouge")


sample_size = min(200, len(test_df))

test_samples = test_df.sample(
    sample_size,
    random_state=42
).reset_index(drop=True)

inputs = test_samples["model_input"].tolist()

targets = test_samples["model_target"].tolist()


predictions = []

batch_size = 16

model.eval()

for i in range(0, len(inputs), batch_size):

    batch_texts = inputs[i:i + batch_size]

    batch_texts = [
        "summarize: " + text
        for text in batch_texts
    ]

    encodings = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **encodings,
            max_length=32,
            num_beams=4,
            early_stopping=True
        )

    batch_preds = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    predictions.extend(batch_preds)


predictions = [
    p.strip()
    for p in predictions
]

targets = [
    t.strip()
    for t in targets
]


rouge_results = rouge.compute(
    predictions=predictions,
    references=targets
)

print("\n===================================")
print("ROUGE RESULTS")
print("===================================\n")

for key, value in rouge_results.items():
    print(f"{key}: {value:.4f}")

exact_matches = [
    pred.lower() == target.lower()
    for pred, target in zip(predictions, targets)
]

exact_match_rate = np.mean(exact_matches)

print("\n===================================")
print(f"Exact Match Rate: {exact_match_rate:.4f}")
print("===================================\n")


results_df = pd.DataFrame({
    "Article": inputs,
    "True Headline": targets,
    "Generated Headline": predictions,
    "Exact Match": exact_matches
})

results_path = "/content/drive/MyDrive/headlineGenerationProjectNLP/flan_t5_evaluation_results.csv"

results_df.to_csv(results_path, index=False)

print(f"✅ Results saved to:\n{results_path}")


print("\n===================================")
print("SAMPLE PREDICTIONS")
print("===================================\n")

for i in range(min(5, len(results_df))):

    print("ARTICLE:")
    print(results_df.loc[i, "Article"][:300])

    print("\nTRUE HEADLINE:")
    print(results_df.loc[i, "True Headline"])

    print("\nGENERATED HEADLINE:")
    print(results_df.loc[i, "Generated Headline"])

    print("\nEXACT MATCH:")
    print(results_df.loc[i, "Exact Match"])

    print("\n" + "=" * 70 + "\n")

Test Samples: 4760

ROUGE RESULTS

rouge1: 0.3182
rouge2: 0.1321
rougeL: 0.2951
rougeLsum: 0.2951

Exact Match Rate: 0.0050

✅ Results saved to:
/content/drive/MyDrive/headlineGenerationProjectNLP/flan_t5_evaluation_results.csv

SAMPLE PREDICTIONS

ARTICLE:
summarize: Recently, The Verge reported on a string of ransomware attacks that have hit cities including Baltimore; Atlanta, Georgia; Newark, New Jersey; and 22 Texas towns. Even The Weather Channel has fallen victim. But before those attacks, there was an attack on the nation's capital, days before

TRUE HEADLINE:
Hackers seized cameras before Trump's inauguration and left a smoking gun behind

GENERATED HEADLINE:
Hackers seized control of Washington, D.C.'s surveillance cameras before inauguration

EXACT MATCH:
False


ARTICLE:
summarize: Many of us are feeling helpless as we watch stories from refugees being detained in airports across the country following President Trump's executive order on Friday afternoon. Some are protestin